# Local GPU student evaluation: Qwen3-0.6B

This notebook runs the same aligned GPQA/SCUA conditions as the API experiment, using a model loaded directly on the GPU server. Every run is saved under `gpu_experiments/outputs/`.

In [1]:
from pathlib import Path
import json
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'gpu_experiments').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'gpu_experiments'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from gpu_experiments import ExperimentConfig, ModelConfig, load_model, preview_task, run_experiment
from student_eval import CONDITION_FILES
print('Project root:', REPO_ROOT)

Project root: /home/f_goodarzi/run_qwen_gpqa/analogy-codex


## Experiment variables

`NUM_ROWS` is the number of aligned questions per condition. Use `None` for all 448 rows. Start with one row and a batch size of one, then increase the batch size to fit the server GPU.

In [2]:
MODEL_ID = 'Qwen/Qwen3-0.6B'
TARGET_CONDITIONS = [0]       # Any subset of 0..8 and 20
NUM_ROWS = 1                  # Rows per condition; None means all rows
START_ROW = 0
BATCH_SIZE = 1                # Increase after the smoke test
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
ENABLE_THINKING = False       # Short JSON mode for the baseline

MODEL_CONFIG = ModelConfig(
    model_id=MODEL_ID,
    dtype='bfloat16',          # Change to float16 if BF16 is unsupported
    device_map='auto',
    attention_implementation=None,  # Or 'flash_attention_2' if installed
    cache_dir=None,
    local_files_only=False,
)
CONFIG = ExperimentConfig(
    model=MODEL_CONFIG,
    condition_ids=tuple(TARGET_CONDITIONS),
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    enable_thinking=ENABLE_THINKING,
)
for condition_id in TARGET_CONDITIONS:
    print(f'{condition_id}: {CONDITION_FILES[condition_id]}')

0: 0_GPQA_free-form_300w_unlimited_deepseek-v4-flash_clean.csv


## Preview one prompt (does not load the model)

In [3]:
preview = preview_task(REPO_ROOT, CONFIG)
print('Request key:', preview['request_key'])
print('Reference answer (not sent):', preview['reference_answer'])
print('\n--- Prompt sent to the model ---\n')
print(preview['prompt'])

Request key: c0:rec055vn3qEqKHHTc
Reference answer (not sent): A

--- Prompt sent to the model ---

You need to select the best answer for a multiple-choice scientific question.
First give a concise reason of no more than 120 words, then choose exactly one
option: A, B, C, or D.
Return only a valid JSON object in this exact form:
{"reason": "your reasoning", "choice": "A"}

This is the question:
A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well 

## Load once

This is the first cell that downloads model files, initializes CUDA, and allocates GPU memory.

In [4]:
loaded_model = load_model(MODEL_CONFIG)
print('Loaded:', loaded_model.config.model_id)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-0.6B


## Run and save

In [5]:
run_dir, results, summary = run_experiment(
    REPO_ROOT, EXPERIMENT_DIR, CONFIG, loaded_model=loaded_model
)
print('Saved run:', run_dir)
print(json.dumps(summary, indent=2))

Completed 1/1
Saved run: /home/f_goodarzi/run_qwen_gpqa/analogy-codex/gpu_experiments/outputs/run_20260817T092411Z
{
  "requested": 1,
  "successful": 1,
  "failed": 0,
  "correct": 0,
  "accuracy": 0.0,
  "failure_rate": 0.0,
  "prompt_tokens": 624,
  "completion_tokens": 45,
  "by_condition": {
    "0": {
      "successful": 1,
      "correct": 0,
      "accuracy": 0.0
    }
  }
}


## Result preview

In [6]:
for result in results[:5]:
    print(json.dumps({
        'request_key': result['request_key'],
        'prediction': result.get('prediction'),
        'reference_answer': result['answer_key'],
        'is_correct': result.get('is_correct'),
        'reason': result.get('reason'),
        'usage': result.get('usage'),
        'latency_seconds': result.get('latency_seconds'),
        'error': result.get('error'),
    }, ensure_ascii=False, indent=2))

{
  "request_key": "c0:rec055vn3qEqKHHTc",
  "prediction": "D",
  "reference_answer": "A",
  "is_correct": false,
  "reason": "The proposed therapy uses a Morpholino to skip an out-of-frame exon, not to repair the gene. The structure not involved is the antisense (D).",
  "usage": {
    "prompt_tokens": 624,
    "completion_tokens": 45
  },
  "latency_seconds": 3.539,
  "error": null
}
